# Freelance Agency Co: colab batch inference
Starts llama-server on localhost, then drains `inference` tasks. This notebook is a batch worker, not a permanent public endpoint: the session is ephemeral.

In [ ]:
# ======== CONFIG: the ONLY cell the model edits. Values come from values/freelance-agency.yaml ========
FIRM_ID = "freelance-agency"
FIRM_REPO = "freelance-agency/freelance-agency"
FIRM_COMMIT = "main"
STATE_REMOTE = "file:///home/z/my-project/workspaces/freelance-agency/state"
HARNESS_URL = "http://localhost:3000/api/harness/firms/freelance-agency"
IDLE_EXIT = 300
MODEL_REPO = "TheBloke/CodeLlama-13B-Instruct-GGUF"
MODEL_FILE = "codellama-13b-instruct.Q4_K_M.gguf"
LLAMA_CPP_REF = "b3906"
CTX = 8192
# Secrets are read from the platform secret store in the next cell. Never type a value here.
# ======== END CONFIG: everything below is identical across firms ========

In [ ]:
import os, subprocess, sys
os.environ['RUN_MODE'] = 'bare'
WORK = '/content'
def get_secret(name):
    from google.colab import userdata
    try:
        return userdata.get(name)
    except Exception:
        return None
for name in ('HF_TOKEN', 'GITHUB_TOKEN'):
    v = get_secret(name)
    if v:
        os.environ[name] = v
missing = [n for n in ('HF_TOKEN',) if not os.environ.get(n)]
if missing:
    raise SystemExit('Add these secrets in the platform secrets panel, then re-run: ' + ', '.join(missing))

FIRM_ROOT = WORK + '/firm'
if not os.path.isdir(FIRM_ROOT):
    tok = os.environ.get('GITHUB_TOKEN')
    url = 'https://' + (tok + '@' if tok else '') + 'github.com/' + FIRM_REPO + '.git'
    subprocess.run(['git', 'clone', url, FIRM_ROOT], check=True)
subprocess.run(['git', '-C', FIRM_ROOT, 'checkout', FIRM_COMMIT], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', FIRM_ROOT + '/requirements.lock'], check=True)
os.environ['FIRM_ROOT'] = FIRM_ROOT
print('bootstrap ok:', FIRM_ID, FIRM_COMMIT)

In [ ]:
STATE_DIR = '/content/state'
os.makedirs(STATE_DIR, exist_ok=True)
os.environ['STATE_DIR'] = STATE_DIR
if STATE_REMOTE.startswith('hf://'):
    # No bucket mount on this platform: mirror state with the hf CLI.
    # Check the exact argument order for your installed version first:
    subprocess.run(['hf', 'sync', '--help'])
    os.environ['STATE_PULL_CMD'] = 'hf sync ' + STATE_REMOTE + ' ' + STATE_DIR
    os.environ['STATE_PUSH_CMD'] = 'hf sync ' + STATE_DIR + ' ' + STATE_REMOTE
os.environ['FIRM_PARTITION'] = os.environ.get('FIRM_PARTITION', 'colab')

In [ ]:
import time, urllib.request
os.environ['MODELS_DIR'] = '/content/models'
os.environ.update(MODEL_REPO=MODEL_REPO, MODEL_FILE=MODEL_FILE, LLAMA_CPP_REF=LLAMA_CPP_REF, FIRM_CTX=str(CTX))
server = subprocess.Popen(['sh', FIRM_ROOT + '/images/inference/run.sh'])
for _ in range(600):
    try:
        urllib.request.urlopen('http://localhost:8000/health', timeout=2)
        break
    except Exception:
        if server.poll() is not None:
            raise SystemExit('llama-server exited; see output above')
        time.sleep(5)
else:
    raise SystemExit('llama-server did not become healthy')
os.environ['INFERENCE_URL'] = 'http://localhost:8000'

In [ ]:
os.environ['FIRM_WORKER_ID'] = FIRM_ID + '-colab-inference'
os.environ['FIRM_IDLE_EXIT'] = str(IDLE_EXIT)
os.environ['HARNESS_BASE_URL'] = HARNESS_URL
try:
    subprocess.run(['sh', FIRM_ROOT + '/images/worker/run.sh'], check=True)
finally:
    server.terminate()